<a href="https://colab.research.google.com/github/heyaankit/pdm-industrial-app/blob/main/notebooks/2_pump_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
# necessary imports

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [4]:
# Loading dataset

df = pd.read_csv('drive/MyDrive/Datasets/PdM Dataset/Large_Industrial_Pump_Maintenance_Dataset.csv')

In [5]:
df.head(3)

,Pump_ID,Temperature,Vibration,Pressure,Flow_Rate,RPM,Operational_Hours,Maintenance_Flag
0,2,127.508350,2.369397,136.021029,6.501492,1444.191922,3966.793672,1
1,4,88.975185,4.541126,147.516973,7.001516,1004.802496,3673.288933,0
2,3,61.832325,2.542112,220.858577,8.824368,2597.662712,5489.061016,1


In [6]:
# Creating classes to predict instead of just predicting maintenance required or not

class_names = [
    'No Failure',
    'Thermal Failure',
    'Hydraulic Failure',
    'Mechanical Failure',
    'Wear-out Failure'
]

In [7]:
n_classes = len(class_names)

In [8]:
df.head(3)

,Pump_ID,Temperature,Vibration,Pressure,Flow_Rate,RPM,Operational_Hours,Maintenance_Flag
0,2,127.508350,2.369397,136.021029,6.501492,1444.191922,3966.793672,1
1,4,88.975185,4.541126,147.516973,7.001516,1004.802496,3673.288933,0
2,3,61.832325,2.542112,220.858577,8.824368,2597.662712,5489.061016,1


In [9]:
# Making thresholds for every feature

temp_high = df['Temperature'].quantile(0.8)
temp_crit = df['Temperature'].quantile(0.9)
vib_high = df['Vibration'].quantile(0.8)
vib_crit = df['Vibration'].quantile(0.9)
pres_high = df['Pressure'].quantile(0.8)
pres_crit = df['Pressure'].quantile(0.9)
flow_low = df['Flow_Rate'].quantile(0.2)
rpm_high = df['RPM'].quantile(0.8)
hrs_high = df['Operational_Hours'].quantile(0.75)
hrs_crit = df['Operational_Hours'].quantile(0.9)

In [10]:
# A Scoring system
# Each pump gets point for different risk factors and totla score determines the failure type.

thermal_score = (
    (df['Temperature'] > temp_high).astype(int) * 2 +
    (df['Temperature'] > temp_crit).astype(int) * 2 +
    (df['RPM'] > rpm_high).astype(int) * 1  # high RPM generates heat
)

mechanical_score = (
    (df['Vibration'] > vib_high).astype(int) * 2 +
    (df['Vibration'] > vib_crit).astype(int) * 2 +
    (df['RPM'] > rpm_high).astype(int) * 1  # high RPM generates vibration
)

hydraulic_score = (
    (df['Pressure'] > pres_high).astype(int) * 2 +
    (df['Flow_Rate'] < flow_low).astype(int) * 3 +  # low flow introduces air cavitation
    (df['Pressure'] > pres_crit).astype(int) * 1
)

wearout_score = (
    (df['Operational_Hours'] > hrs_high).astype(int) * 2 +
    (df['Operational_Hours'] > hrs_crit).astype(int) * 3 +
    (df['Vibration'] > vib_high).astype(int) * 1 +   # Old pumps vibrate more
    (df['Vibration'] > vib_crit).astype(int) * 1
)

# Minimum score needed to classify as a failure
threshold = 3

# Creating failure flag
thermal_flag = (thermal_score >= threshold).astype(int)
mechanical_flag = (mechanical_score >= threshold).astype(int)
hydraulic_flag = (hydraulic_score >= threshold).astype(int)
wearout_flag = (wearout_score >= threshold).astype(int)

In [11]:
# Priority-based failure system
# When a pump has multiple failures, it returns the most critical one first

def priority_failure(row):
  mechanical = row['mechanical_flag']
  thermal = row['thermal_flag']
  hydraulic = row['hydraulic_flag']
  wearout = row['wearout_flag']

  if mechanical:
    return 1
  elif thermal:
    return 1
  elif hydraulic:
    return 1
  elif wearout:
    return 1
  else:
    return 0

# Appending new columns to the dataframe

df['mechanical_flag'] = mechanical_flag
df['thermal_flag'] = thermal_flag
df['hydraulic_flag'] = hydraulic_flag
df['wearout_flag'] = wearout_flag

# This apply function to every row in the dataframe, this will be our target variable

df['Failure_Type'] = df.apply(priority_failure, axis = 1)

In [12]:
df.head(2)

,Pump_ID,Temperature,Vibration,Pressure,Flow_Rate,RPM,Operational_Hours,Maintenance_Flag,mechanical_flag,thermal_flag,hydraulic_flag,wearout_flag,Failure_Type
0,2,127.508350,2.369397,136.021029,6.501492,1444.191922,3966.793672,1,0,0,0,0,0
1,4,88.975185,4.541126,147.516973,7.001516,1004.802496,3673.288933,0,1,0,0,0,1


In [13]:
# Needs Domain knowldege to create new features

# Interaction Features

# Temperature and Vibration
df['Temperature_x_Vibration'] = df['Temperature'] * df['Vibration']

# Pressure and Flow
df['Pressure_x_Flow'] = df['Pressure'] * df['Flow_Rate']

# RPM and Vibration
df['RPM_x_Vibration'] = df['RPM'] * df['Vibration']

# Temperature and RPM
df['Temperature_x_RPM'] = df['Temperature'] * df['RPM']


# Ratio Features

# Pressure Flow ratio
df['Pressure_flow_ratio'] = df['Pressure'] / (df['Flow_Rate'] + 1e-6)

# RPM per Hour
df['RPM_per_hour'] = df['RPM'] / (df['Operational_Hours'] + 1e-6)

# Power Proxy
df['Power_proxy'] = (df['Pressure'] * df['Flow_Rate']) / 1000


# Log Transform Features (some pumps are brand new 10 hrs, but others have operated for 10,000 hours)

# Log vibration
df['Log_vibration'] = np.log1p(df['Vibration'])

# Log operational time
df['Log_operational_hrs'] = np.log1p(df['Operational_Hours'])


# Polynomial Features

# Temperature square (heat deamage accelerates at higher temp)
df['Temperature_sq'] = df['Temperature']**2

# Vibration square
df['Vibration_sq'] = df['Vibration']**2


# Risk bin features

# Operational hours
df['Hours_bin'] = pd.cut(
    df['Operational_Hours'],
    bins = [0, 2500, 5000, 7500, 10001],
    labels = [1, 2, 3, 4]
)

# Temperature
df['Temperature_bin'] = pd.cut(
    df['Temperature'],
    bins = [0, 75, 100, 125, 151],
    labels = [1, 2, 3, 4]
)


In [14]:
# Dropping the newly created flag columns, it's role is served (use to make failure type target variable)

df = df.drop(columns=['mechanical_flag', 'thermal_flag', 'hydraulic_flag', 'wearout_flag'])


In [15]:
df.head(3)

,Pump_ID,Temperature,Vibration,Pressure,Flow_Rate,RPM,Operational_Hours,Maintenance_Flag,Failure_Type,Temperature_x_Vibration,...,Temperature_x_RPM,Pressure_flow_ratio,RPM_per_hour,Power_proxy,Log_vibration,Log_operational_hrs,Temperature_sq,Vibration_sq,Hours_bin,Temperature_bin
0,2,127.508350,2.369397,136.021029,6.501492,1444.191922,3966.793672,1,0,302.117889,...,184146.529119,20.921507,0.364070,0.884340,1.214734,8.285965,16258.379340,5.614042,2,4
1,4,88.975185,4.541126,147.516973,7.001516,1004.802496,3673.288933,0,1,404.047519,...,89402.488009,21.069287,0.273543,1.032842,1.712198,8.209115,7916.583556,20.621825,2,2
2,3,61.832325,2.542112,220.858577,8.824368,2597.662712,5489.061016,1,0,157.184691,...,160619.523933,25.028257,0.473244,1.948937,1.264723,8.610695,3823.236363,6.462333,3,1


### Preprocessing

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   Pump_ID                  20000 non-null  int64   
 1   Temperature              20000 non-null  float64 
 2   Vibration                20000 non-null  float64 
 3   Pressure                 20000 non-null  float64 
 4   Flow_Rate                20000 non-null  float64 
 5   RPM                      20000 non-null  float64 
 6   Operational_Hours        20000 non-null  float64 
 7   Maintenance_Flag         20000 non-null  int64   
 8   Failure_Type             20000 non-null  int64   
 9   Temperature_x_Vibration  20000 non-null  float64 
 10  Pressure_x_Flow          20000 non-null  float64 
 11  RPM_x_Vibration          20000 non-null  float64 
 12  Temperature_x_RPM        20000 non-null  float64 
 13  Pressure_flow_ratio      20000 non-null  float64 
 14  RPM_pe

In [17]:
# dropping cols for processing
X = df.drop(columns=['Pump_ID', 'Maintenance_Flag', 'Failure_Type'], axis=1)
y = df['Failure_Type']

In [18]:
# train test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [19]:
# Scaling all the columns

scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [20]:
# Binarize the categorical target columns

y_train = label_binarize(y_train, classes = list(range(n_classes)))
y_test = label_binarize(y_test, classes = list(range(n_classes)))

### Training

In [23]:
model = DecisionTreeClassifier(
        max_depth = 10,
        min_samples_leaf = 15,
        class_weight = 'balanced',
        random_state = 42
    )



In [24]:
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

In [27]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {acc}")

Accuracy Score: 0.9975


In [28]:
import pickle

model_filename = 'decision_tree_model.pkl'

with open(model_filename, 'wb') as file:
    pickle.dump(model, file)

print(f"Model successfully saved to {model_filename}")

Model successfully saved to decision_tree_model.pkl
